# ND-AWS amplitude-damping analysis — the simulated damping sweep

Reads the datasets the 20-qubit amplitude-damping notebook produces and draws one panel per
Hamiltonian class: the mean convergence of the noise-directed and the standard warm-started
ansatz at each damping strength, against the noiseless simulation.

The main noise-directed-against-standard study is analyzed separately, in
`00b_ndaws_main_analysis.ipynb`.

Which datasets enter which panel is stated explicitly in the configuration cell — there is
no implicit dataset discovery. Reading, exclusions and panel styling come from
`helpers/ndaws_paper_analysis.py`, which both analysis notebooks share, so run this notebook
from its own folder.

**Writes**: `fig2_<panel>.pdf` into `./temp/figures` inside the notebook folder. Nothing is
written into the result store.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import plotly.io
from plotly.colors import sequential
from plotly.graph_objs import Scatter
from plotly.subplots import make_subplots

from helpers.ndaws_paper_analysis import (AR_FLOOR,
                                          FIGURE_DIR,
                                          LINE_WIDTH_BORDER,
                                          finish_panel,
                                          load_dataset,
                                          pad_to_common_length)

## Panel specifications

The sweep notebook builds every dataset name it writes from one `RUN_ID`. The cell below
applies the same rule in reverse, so pointing this notebook at a sweep means setting
`RUN_ID` to the value that sweep was written under — no dataset name is typed out by hand
and no dataset is discovered implicitly.

`FIG2_PANELS` ends up holding, per Hamiltonian class, the noiseless dataset, one dataset per
damping strength for each ansatz, and the `backend` whose folder they live in. Anything the
sweep notebook produces lives under `aer`.

Styling keys off the dataset names themselves: `no-gauges-version` selects the standard
ansatz, `AMP=` the damping colour scale. A colour scale of six steps is used, so keep
`Q_VALUES` to at most six damping strengths.

In [ ]:
# Set this to the RUN_ID the sweep was written under.
RUN_ID = 'run-01'

BACKEND_NAME = 'aer'

Q_VALUES = [0.01, 0.02, 0.03, 0.04, 0.05, 0.1]

# Panel key -> the class token the sweep notebook writes into the name.
FIG2_CLASSES = {'ER-10': 'ER-0.1',
                'ER-20': 'ER-0.2',
                'RG-3': 'RG-3'}


def build_dataset_name(class_token, q, ansatz, run_id=RUN_ID):
    """The naming rule of `01a_ndaws_amplitude_damping_run.ipynb`, read back.

    That notebook runs at `time_block_size = 1.0` and `add_hdls = False`, so neither the
    `k=` nor the `HDLS_` token appears in a name. Change this alongside it if either
    setting changes there.
    """
    name = 'NDAWS_simulation_'
    name += 'noiseless_' if q == 0 else 'noisy_'
    name += f'{class_token}_'
    if q != 0:
        name += f'NM=amplitude-damping_AMP={q}_'
    if ansatz == 'Standard':
        name += 'no-gauges-version_'
    return name + run_id


FIG2_PANELS = {
    panel_key: {'noiseless': build_dataset_name(class_token, 0.0, 'ND'),
                'nd': {q: build_dataset_name(class_token, q, 'ND') for q in Q_VALUES},
                'standard': {q: build_dataset_name(class_token, q, 'Standard') for q in Q_VALUES},
                'backend': BACKEND_NAME}
    for panel_key, class_token in FIG2_CLASSES.items()
}

# A member written under a different RUN_ID is overridden here, after the dict is built:
#   FIG2_PANELS['RG-3']['nd'][0.02] = build_dataset_name('RG-3', 0.02, 'ND', run_id='run-04')

for _panel_key, _spec in FIG2_PANELS.items():
    print(f"{_panel_key:<6} {_spec['noiseless']}")

os.makedirs(FIGURE_DIR, exist_ok=True)

## Plotting

The panels draw `1 - AR` on a reversed logarithmic axis, so better is up, with a floor at
`1e-3`. A run shorter than the longest in its panel is padded with its own best
approximation ratio.

Means only, one line per damping strength and ansatz: reds with crosses for the standard
ansatz, blues with circles for the noise-directed one, and the noiseless simulation in a
single solid line. Each point averages the ten instances and all three seeded runs of each
instance — unlike the 100-qubit study, which reports the best of the three. The axes, the
fonts and the padding rule come from the shared module.

In [ ]:
def plot_fig2_panel(panel_key, subfig_letter, loaded_members, q_of_dataset):
    merged = make_subplots(rows=1, cols=1)
    max_x_global = np.max([np.max([len(ys) for ys in m['all_ys']]) for m in loaded_members])

    unique_qs = sorted({q for q in q_of_dataset.values() if q is not None})
    unique_qs_enumerate_rev = {v: k for k, v in enumerate(unique_qs)}

    for member in loaded_members:
        dataset_name = member['dataset_name']
        q = q_of_dataset[dataset_name]

        if q is None:
            color_main = 'rgba(65, 105, 225)'
            symbol_marker = None
            name_plot = "Noiseless"
        elif 'no-gauges-version' in dataset_name:
            color_main = 'rgba' + sequential.Reds_r[unique_qs_enumerate_rev[q]][3:]
            symbol_marker = 'x'
            name_plot = f"q={q} (S)"
        else:
            color_main = 'rgba' + sequential.deep[1:][0:6][::-1][unique_qs_enumerate_rev[q]][3:]
            symbol_marker = 'circle'
            name_plot = f"q={q} (ND)"

        color_main_1 = color_main[:-1] + ', 1.0)'

        all_ys_extended = 1. - pad_to_common_length(member['all_ys'])
        all_ys_extended = np.clip(all_ys_extended, AR_FLOOR, None)

        xs = list(range(0, all_ys_extended.shape[1]))
        mean_ys = np.mean(all_ys_extended, axis=0)

        merged.add_trace(Scatter(x=xs,
                                 y=mean_ys,
                                 mode='lines' if q is None else 'markers+lines',
                                 line_dash='dash' if 'no-gauges-version' in dataset_name else 'solid',
                                 line_width=LINE_WIDTH_BORDER,
                                 line_color=color_main_1,
                                 marker_size=LINE_WIDTH_BORDER * 2,
                                 marker_symbol=symbol_marker,
                                 name=name_plot,
                                 legendgroup=name_plot,
                                 showlegend=True))

    merged.update_layout(xaxis_title=r"Iteration", yaxis_title=r"1-AR (log scale)")
    merged.update_xaxes(range=(-0.5, max_x_global + 1))

    return finish_panel(merged,
                        panel_key=panel_key,
                        subfig_letter=subfig_letter,
                        yrange=np.log10(np.array([0.0025, 0.25][::-1])).tolist(),
                        xticks=np.linspace(0, 15, 6, endpoint=True).astype(int),
                        xrange=(-1, 16),
                        height=int(1600 * 0.9),
                        width=int(2600 * 0.7))

## Read the damping sweep and draw it

In [ ]:
FIG2_DATA = {}

# Every run enters the average here, not the best of the three per instance: the damping
# panels report the mean over the ten instances and their three seeded runs.
INCLUDE_ONLY_BEST_RUN = False

for _index, (panel_key, spec) in enumerate(FIG2_PANELS.items()):
    print('\n' + '=' * 30, panel_key, '=' * 30)
    members = []
    q_of_dataset = {}

    noiseless_name = spec.get('noiseless')
    if noiseless_name is not None:
        loaded = load_dataset(dataset_name=noiseless_name,
                              backend_name=spec['backend'],
                              simulation=True,
                              include_only_best_run=INCLUDE_ONLY_BEST_RUN)
        loaded['role'] = 'noiseless'
        q_of_dataset[noiseless_name] = None
        members.append(loaded)

    for role in ('nd', 'standard'):
        for q, dataset_name in sorted(spec.get(role, {}).items()):
            loaded = load_dataset(dataset_name=dataset_name,
                                  backend_name=spec['backend'],
                                  simulation=True,
                                  include_only_best_run=INCLUDE_ONLY_BEST_RUN)
            loaded['role'] = role
            q_of_dataset[dataset_name] = q
            members.append(loaded)

    FIG2_DATA[panel_key] = members

    figure = plot_fig2_panel(panel_key=panel_key,
                             subfig_letter='abcdefg'[_index],
                             loaded_members=members,
                             q_of_dataset=q_of_dataset)
    file_path = os.path.join(FIGURE_DIR, f'fig2_{panel_key}.pdf')
    plotly.io.write_image(figure, file_path, format='pdf', scale=1)
    print('wrote', file_path)